In [3]:
%pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.2 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip -q install mne numpy scipy scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

path = Path.cwd() / "DataPaper" / "user_1" / "Actigraph.csv"
print(path)
print(path.exists())

df = pd.read_csv(path)

df.head()

/Users/justinbanh/Documents/Github/sleep-wearable/tests/DataPaper/user_1/Actigraph.csv
True


,Unnamed: 0,Axis1,Axis2,Axis3,Steps,HR,Inclinometer Off,Inclinometer Standing,Inclinometer Sitting,Inclinometer Lying,Vector Magnitude,day,time
0,82,0,0,0,0,68.0,0,1,0,0,0.00,1,10:10:22
1,83,11,4,7,1,68.0,0,1,0,0,13.64,1,10:10:23
2,84,0,21,10,0,68.0,0,0,1,0,23.26,1,10:10:24
3,85,0,1,24,0,68.0,0,0,1,0,24.02,1,10:10:25
4,86,34,14,63,1,154.0,0,1,0,0,72.95,1,10:10:26


In [4]:
import numpy as np

# quick check
print(df.columns)
print(len(df))

# compute linear acceleration magnitude from raw axes
df["acc_mag"] = np.sqrt(
    df["Axis1"]**2 +
    df["Axis2"]**2 +
    df["Axis3"]**2
)


Index(['Unnamed: 0', 'Axis1', 'Axis2', 'Axis3', 'Steps', 'HR',
       'Inclinometer Off', 'Inclinometer Standing', 'Inclinometer Sitting',
       'Inclinometer Lying', 'Vector Magnitude', 'day', 'time'],
      dtype='str')
67936


In [5]:
# combine day + time into datetime
import pandas as pd

ts = pd.to_datetime(df["day"].astype(str) + " " + df["time"].astype(str))
dt = ts.diff().dt.total_seconds().median()

fs = round(1 / dt)
fs


/var/folders/k5/5h4jxztd6l5cmllxr_fx5jw40000gn/T/ipykernel_21207/2219792890.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df["day"].astype(str) + " " + df["time"].astype(str))


1

In [10]:
epoch_sec = 30
epoch_len = fs * epoch_sec


In [11]:
sleep = pd.read_csv("DataPaper/user_1/Sleep.csv")
print(sleep.head())


   Unnamed: 0  In Bed Date In Bed Time  Out Bed Date Out Bed Time  Onset Date  \
0           0            2       00:46             2        03:31           2   
1           1            2       03:57             2        07:30           2   

  Onset Time  Latency  Efficiency  Total Minutes in Bed  \
0      00:46        0       87.27                   165   
1      03:57        0       92.02                   213   

   Total Sleep Time (TST)  Wake After Sleep Onset (WASO)  \
0                     144                             21   
1                     196                             17   

   Number of Awakenings  Average Awakening Length  Movement Index  \
0                     9                      2.33           9.091   
1                     9                      1.89           8.920   

   Fragmentation Index  Sleep Fragmentation Index  
0                   10                     19.091  
1                    0                      8.920  


In [13]:
import pandas as pd

# combine day + time into datetime
df["timestamp"] = pd.to_datetime(
    df["day"].astype(str) + " " + df["time"].astype(str),
    errors="coerce"
)

df[["timestamp"]].head()



/var/folders/k5/5h4jxztd6l5cmllxr_fx5jw40000gn/T/ipykernel_21207/3324717639.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(


,timestamp
0,1-01-01 10:10:22
1,1-01-01 10:10:23
2,1-01-01 10:10:24
3,1-01-01 10:10:25
4,1-01-01 10:10:26


In [14]:
sleep_intervals = []

for _, row in sleep.iterrows():
    start = pd.to_datetime(
        f"{row['In Bed Date']} {row['In Bed Time']}",
        errors="coerce"
    )
    end = pd.to_datetime(
        f"{row['Out Bed Date']} {row['Out Bed Time']}",
        errors="coerce"
    )
    sleep_intervals.append((start, end))

sleep_intervals


[(Timestamp('1-01-02 00:46:00'), Timestamp('1-01-02 03:31:00')),
 (Timestamp('1-01-02 03:57:00'), Timestamp('1-01-02 07:30:00'))]

In [15]:
def is_sleep(ts):
    for start, end in sleep_intervals:
        if start <= ts <= end:
            return 1
    return 0

df["sleep_label"] = df["timestamp"].apply(is_sleep)

df["sleep_label"].value_counts()


sleep_label
0    49768
1    18168
Name: count, dtype: int64

In [16]:
acc = df["acc_mag"].values
labels = df["sleep_label"].values

X_raw = []
y = []

for i in range(0, len(acc) - epoch_len, epoch_len):
    X_raw.append(acc[i:i + epoch_len])
    y.append(labels[i:i + epoch_len].max())  # any sleep = sleep

X_raw = np.array(X_raw)
y = np.array(y)

X_raw.shape, y.shape, y.mean()


((67935, 1), (67935,), np.float64(0.2674321042172665))

In [17]:
from scipy.stats import iqr
from scipy.signal import welch

def extract_features(x, fs):
    f, psd = welch(x, fs=fs, nperseg=min(256, len(x)))
    return [
        np.mean(x),
        np.std(x),
        np.var(x),
        np.median(x),
        iqr(x),
        np.sqrt(np.mean(x**2)),      # RMS
        np.sum(np.abs(np.diff(x))),  # activity
        np.sum(psd),
        f[np.argmax(psd)]
    ]

X = np.array([extract_features(w, fs) for w in X_raw])
X.shape


(67935, 9)

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(Xtr, ytr)
pred = rf.predict(Xte)

print(confusion_matrix(yte, pred))
print(classification_report(yte, pred, digits=3))


[[5635 4318]
 [  43 3591]]
              precision    recall  f1-score   support

           0      0.992     0.566     0.721      9953
           1      0.454     0.988     0.622      3634

    accuracy                          0.679     13587
   macro avg      0.723     0.777     0.672     13587
weighted avg      0.848     0.679     0.695     13587

